# Oppgave 1: Geokoding (10 poeng)

Det overordnede målet med oppgavene a)-e) er å finne ut **hvor mange mennesker som bor innen gangavstand (1,5 km) fra visse kjøpesentre i Oslo**.

Oppgave 1 gjelder lokasjonene til kjøpesentrene: finn adressene til kjøpesenterne og konverter de til koordinater.

In [43]:
# Imports
import pathlib
import pandas as pd
import geopandas as gpd

### a) Forbered en inputfil som inneholder adressene til kjøpesentre

Finn ut adressene til følgende kjøpesentre (f.eks. ved å bruke din favorittsøkemotor), og samle dem i en tekstfil kalt `shopping_centres.txt`:

 - Oslo City
 - Bryn Senter
 - Storo Storsenter
 - Lambertseter senter
 - Manglerud Senter
 - Linderud senter
 - Tveita Senter
 
Tekstfilen skal være i semikolon-separert format (`;`) og inkludere følgende kolonner:

- `id` (integer) en unik identifikator for hvert kjøpesenter
- `navn` (string) navnet på hvert kjøpesenter
- `adr` (string) adressen

Se et eksempel på hvordan du formaterer tekstfilen [fra forelesningen](https://haavardaagesen.github.io/gmgi221/content/notebooks/04_geokoding-i-geopandas.html).


### b) Les in listen med adresser

Les inn listen med adresser du nettopp forberedte inn i en `pandas.DataFrame` kalt `shopping_centres`

In [68]:
# SKRIV DIN KODE HER OG FJERN LINJEN "raise NotImplementedError()"

NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_MAPPE = NOTEBOOK_PATH / "data"

shopping_centres = pd.read_csv(DATA_MAPPE / "shopping_centres.txt",sep=";")
shopping_centres.head()

,id,navn,adr
0,1,Oslo City,"Stenersgata 1, Sentrum, 0050 Oslo"
1,2,Bryn Senter,"Østensjøveien 79, Østensjø, 0667 Oslo"
2,3,Stor Storsenter,"Vitaminveien 7, Sagene, 0485 Oslo"
3,4,Lambertseter Senter,"Cecilie Thoresens vei 17, Nordstrand, 1153 Oslo"
4,5,Manglerud Senter,"Plogveien 6, Østensjø, 0679 Oslo"


In [69]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION

### c) Geokode adressene

Du skal nå geokode adressene ved å bruke Nominatim sin geokodingstjenesten. Slå sammen resultatene med inputdataene, og lagre dem i en `geopandas.GeoDataFrame` med samme navn (`shopping_centres`).

Husk å definere en tilpasset `user_agent`-streng!

In [70]:
# SKRIV DIN KODE HER OG FJERN LINJEN "raise NotImplementedError()"

shopping_centres = shopping_centres.join(gpd.tools.geocode(
    shopping_centres["adr"],
    provider="nominatim",
    user_agent="simkro",
    timeout=10
))

shopping_centres = gpd.GeoDataFrame(shopping_centres, geometry="geometry", crs="EPSG:4326")
shopping_centres.head()

,id,navn,adr,geometry,address
0,1,Oslo City,"Stenersgata 1, Sentrum, 0050 Oslo",POINT (10.75157 59.91275),"1B, Stenersgata, Vaterland, Sentrum, Oslo, 005..."
1,2,Bryn Senter,"Østensjøveien 79, Østensjø, 0667 Oslo",POINT (10.82246 59.90319),"79, Østensjøveien, Rognerud, Østensjø, Oslo, 0..."
2,3,Stor Storsenter,"Vitaminveien 7, Sagene, 0485 Oslo",POINT (10.77539 59.94684),"7, Vitaminveien, Storo, Sagene, Oslo, 0485, Norge"
3,4,Lambertseter Senter,"Cecilie Thoresens vei 17, Nordstrand, 1153 Oslo",POINT (10.81046 59.87491),"17, Cecilie Thoresens vei, Karlsrud, Nordstran..."
4,5,Manglerud Senter,"Plogveien 6, Østensjø, 0679 Oslo",POINT (10.81286 59.89729),"6, Plogveien, Manglerud, Østensjø, Oslo, 0679,..."


In [71]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION

Sjekk at koordinatsystemet til det geokodede resultatet er korrekt definert, og **reprojiser laget til ETRS89** (EPSG:25832):

In [72]:
# SKRIV DIN KODE HER OG FJERN LINJEN "raise NotImplementedError()"

shopping_centres.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [73]:
shopping_centres = shopping_centres.to_crs("EPSG:25832")
shopping_centres.head()

,id,navn,adr,geometry,address
0,1,Oslo City,"Stenersgata 1, Sentrum, 0050 Oslo",POINT (597948.012 6642990.016),"1B, Stenersgata, Vaterland, Sentrum, Oslo, 005..."
1,2,Bryn Senter,"Østensjøveien 79, Østensjø, 0667 Oslo",POINT (601940.781 6642032.843),"79, Østensjøveien, Rognerud, Østensjø, Oslo, 0..."
2,3,Stor Storsenter,"Vitaminveien 7, Sagene, 0485 Oslo",POINT (599177.887 6646821.293),"7, Vitaminveien, Storo, Sagene, Oslo, 0485, Norge"
3,4,Lambertseter Senter,"Cecilie Thoresens vei 17, Nordstrand, 1153 Oslo",POINT (601355.975 6638866.012),"17, Cecilie Thoresens vei, Karlsrud, Nordstran..."
4,5,Manglerud Senter,"Plogveien 6, Østensjø, 0679 Oslo",POINT (601421.991 6641360.986),"6, Plogveien, Manglerud, Østensjø, Oslo, 0679,..."


In [74]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION

### d) Opprett en *buffer* rundt punktene

Beregn en 1,5 km buffer for hvert geokodede punkt. Overskriv `geometry`-kolonnen med den nye buffergeometrien.

Bruk [`geopandas.GeoDataFrame.buffer()`-metoden](http://geopandas.org/geometric_manipulations.html#GeoSeries.buffer), som bruker shapely’s [`buffer()`](http://toblerity.org/shapely/manual.html#object.buffer) i bakgrunnen. Du trenger bare å bry deg om `distance`-parameteren, ikke bekymre deg for de andre mulige argumentene.

In [75]:
# SKRIV DIN KODE HER OG FJERN LINJEN "raise NotImplementedError()"

shopping_centres["geometry"] = shopping_centres.buffer(distance=1500)
shopping_centres.head()

,id,navn,adr,geometry,address
0,1,Oslo City,"Stenersgata 1, Sentrum, 0050 Oslo","POLYGON ((599448.012 6642990.016, 599440.789 6...","1B, Stenersgata, Vaterland, Sentrum, Oslo, 005..."
1,2,Bryn Senter,"Østensjøveien 79, Østensjø, 0667 Oslo","POLYGON ((603440.781 6642032.843, 603433.558 6...","79, Østensjøveien, Rognerud, Østensjø, Oslo, 0..."
2,3,Stor Storsenter,"Vitaminveien 7, Sagene, 0485 Oslo","POLYGON ((600677.887 6646821.293, 600670.664 6...","7, Vitaminveien, Storo, Sagene, Oslo, 0485, Norge"
3,4,Lambertseter Senter,"Cecilie Thoresens vei 17, Nordstrand, 1153 Oslo","POLYGON ((602855.975 6638866.012, 602848.752 6...","17, Cecilie Thoresens vei, Karlsrud, Nordstran..."
4,5,Manglerud Senter,"Plogveien 6, Østensjø, 0679 Oslo","POLYGON ((602921.991 6641360.986, 602914.769 6...","6, Plogveien, Manglerud, Østensjø, Oslo, 0679,..."


In [76]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION

### e) Lagre buffergeometrilaget

Lagre dataframen som inneholder buffergeometriene i en *GeoJSON*-fil med navn `shopping_centres.geojson`:

In [ ]:
# SKRIV DIN KODE HER OG FJERN LINJEN "raise NotImplementedError()"

shopping_centres.to_file(DATA_MAPPE / 'shopping_centres.geojson', driver='GeoJSON')

In [78]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION

## Ferdig!

Supert, da er du ferdig med denne øvingsoppaven.